# Warden toxicity model

PyTorch inference baseline for the locally downloaded `unitary/toxic-bert` model, followed by an optional fine-tuning scaffold. The model is multi-label, so one post can receive several toxicity categories.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_PATH = Path('../models/toxic-bert')
LABELS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if not MODEL_PATH.exists():
    raise FileNotFoundError(f'Model not found: {MODEL_PATH.resolve()}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH, local_files_only=True, use_safetensors=False
).to(DEVICE)
model.eval()

print('device:', DEVICE)
print('labels:', LABELS)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

device: cuda
labels: ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']


In [2]:
def score_text(text: str) -> dict[str, float]:
    if not isinstance(text, str) or not text.strip():
        raise ValueError('text must be a non-empty string')

    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
    inputs = {key: value.to(DEVICE) for key, value in inputs.items()}
    with torch.inference_mode():
        logits = model(**inputs).logits[0]
        probabilities = torch.sigmoid(logits).cpu().numpy()
    return {label: round(float(probabilities[i]), 4) for i, label in enumerate(LABELS)}

def classify_text(text: str, warning_threshold=0.40, hide_threshold=0.75) -> dict:
    scores = score_text(text)
    toxicity_score = scores['toxic']
    action = 'hide' if toxicity_score >= hide_threshold else 'warn' if toxicity_score >= warning_threshold else 'show'
    return {'text': text, 'toxicity_score': toxicity_score, 'categories': scores, 'action': action}

In [11]:
examples = [
    'I disagree with your point, but here is another perspective.',
    'You are disgusting and nobody likes you.',
    'I will find you and make you regret this.',
    'You are a complete idiot and should be ashamed of yourself.',
    'Resign and enjoy your life why all this bullshirt job want respect want money want but no work'
]

for text in examples:
    print(json.dumps(classify_text(text), indent=2))
    print()

{
  "text": "I disagree with your point, but here is another perspective.",
  "toxicity_score": 0.0006,
  "categories": {
    "toxic": 0.0006,
    "severe_toxic": 0.0001,
    "obscene": 0.0002,
    "threat": 0.0001,
    "insult": 0.0002,
    "identity_hate": 0.0001
  },
  "action": "show"
}

{
  "text": "You are disgusting and nobody likes you.",
  "toxicity_score": 0.98,
  "categories": {
    "toxic": 0.98,
    "severe_toxic": 0.0101,
    "obscene": 0.2346,
    "threat": 0.0016,
    "insult": 0.8707,
    "identity_hate": 0.0183
  },
  "action": "hide"
}

{
  "text": "I will find you and make you regret this.",
  "toxicity_score": 0.3686,
  "categories": {
    "toxic": 0.3686,
    "severe_toxic": 0.0067,
    "obscene": 0.0056,
    "threat": 0.313,
    "insult": 0.0075,
    "identity_hate": 0.0087
  },
  "action": "show"
}

{
  "text": "You are a complete idiot and should be ashamed of yourself.",
  "toxicity_score": 0.9823,
  "categories": {
    "toxic": 0.9823,
    "severe_toxic": 0.0

## Optional fine-tuning

Place a labelled CSV at `../data/toxicity/train.csv`. It must contain `comment_text` plus the six columns in `LABELS`, each containing 0/1 values. Run the next cells only after inspecting the dataset.

In [4]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from transformers import DataCollatorWithPadding, TrainingArguments, Trainer

DATA_PATH = Path('../data/toxicity/train.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Add a labelled dataset at {DATA_PATH.resolve()}')

df = pd.read_csv(DATA_PATH)
required = {'comment_text', *LABELS}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f'Missing dataset columns: {sorted(missing)}')
df = df[['comment_text', *LABELS]].dropna(subset=['comment_text']).copy()
# Fast first experiment: use 30k examples, then scale up after the pipeline works.
df = df.sample(n=min(30000, len(df)), random_state=42).reset_index(drop=True)
for label in LABELS:
    df[label] = df[label].astype(np.float32)
train_df, valid_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['toxic'].astype(int))

class ToxicityDataset(Dataset):
    def __init__(self, frame):
        self.encodings = tokenizer(
            frame['comment_text'].tolist(), truncation=True, max_length=192
        )
        self.labels = torch.tensor(frame[LABELS].to_numpy(), dtype=torch.float32)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        item = {key: values[index] for key, values in self.encodings.items()}
        item['labels'] = self.labels[index]
        return item

train_dataset = ToxicityDataset(train_df)
valid_dataset = ToxicityDataset(valid_df)
finetune_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH, local_files_only=True, use_safetensors=False,
    num_labels=len(LABELS), problem_type='multi_label_classification'
).to(DEVICE)
finetune_model.config.id2label = dict(enumerate(LABELS))
finetune_model.config.label2id = {label: i for i, label in enumerate(LABELS)}

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [5]:
def compute_metrics(prediction):
    from sklearn.metrics import f1_score, roc_auc_score
    logits, labels = prediction
    probabilities = 1 / (1 + np.exp(-logits))
    predicted = (probabilities >= 0.5).astype(int)
    return {
        'macro_f1': f1_score(labels, predicted, average='macro', zero_division=0),
        'macro_roc_auc': roc_auc_score(labels, probabilities, average='macro'),
    }

# Transformers renamed evaluation_strategy to eval_strategy in newer releases.
import inspect
strategy_argument = ('eval_strategy' if 'eval_strategy' in inspect.signature(TrainingArguments).parameters else 'evaluation_strategy')
training_kwargs = dict(
    output_dir='../models/toxic-bert-finetuned',
    learning_rate=2e-5, per_device_train_batch_size=16,
    per_device_eval_batch_size=32, num_train_epochs=2,
    fp16=torch.cuda.is_available(),
    weight_decay=0.01, save_strategy='epoch',
    load_best_model_at_end=True, metric_for_best_model='macro_roc_auc',
    report_to='none'
)
training_kwargs[strategy_argument] = 'epoch'
training_args = TrainingArguments(**training_kwargs)

class ToxicityCollator:
    def __init__(self, tokenizer):
        self.padding = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors='pt')

    def __call__(self, features):
        features = [dict(feature) for feature in features]
        labels = torch.stack([torch.as_tensor(feature.pop('labels'), dtype=torch.float32) for feature in features])
        batch = self.padding(features)
        batch['labels'] = labels
        return batch

data_collator = ToxicityCollator(tokenizer)
trainer = Trainer(model=finetune_model, args=training_args,
                  train_dataset=train_dataset, eval_dataset=valid_dataset,
                  data_collator=data_collator,
                  compute_metrics=compute_metrics)

# Uncomment only after reviewing the dataset:
# trainer.train()
# print(trainer.evaluate())
# trainer.save_model('../models/toxic-bert-finetuned')
# tokenizer.save_pretrained('../models/toxic-bert-finetuned')

In [6]:
# Uncomment only after reviewing the dataset:
trainer.train()
print(trainer.evaluate())
trainer.save_model('../models/toxic-bert-finetuned')
tokenizer.save_pretrained('../models/toxic-bert-finetuned')

Epoch,Training Loss,Validation Loss,Macro F1,Macro Roc Auc
1,0.027426,0.024948,0.764559,0.996233
2,0.016177,0.024684,0.788206,0.996037


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Macro F1,Macro Roc Auc
0.016177,0.024948,2,0.764559,0.996233


{'eval_loss': 0.024947596713900566, 'eval_macro_f1': 0.7645593055288588, 'eval_macro_roc_auc': 0.9962327571076283}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('../models/toxic-bert-finetuned\\tokenizer_config.json',
 '../models/toxic-bert-finetuned\\tokenizer.json')